## Setup 

In [1]:
import sys 
import os
import IPython


# Env. vars to check if the notebook is running on colab, kaggle etc. 
IS_COLAB = "google.colab" in sys.modules 
IS_KAGGLE = "kaggle_secrets" in sys.modules 
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

if IS_COLAB:
    # Install the packages 
    %pip install -q -U tensorflow-addons
    %pip install -q -U transformers
    %pip install -q -U datasets
    print("You can safely ignore the package incompatibility errors.")
    # Mount the drive 
    from google.colab import drive 
    drive.mount("/drive")

# NOTE: This needs to be done for correct notebook behavior with the transformers 
# library 
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [2]:
import pandas as pd
import numpy as np
from tqdm import tqdm 
from copy import deepcopy 

import random 
import shutil 
# Scikit-Learn ≥0.20 is required
import sklearn
assert sklearn.__version__ >= "0.20"

# Pytorch imports 
import torch
import torch.nn as nn 
from torch.utils.data import Dataset, DataLoader

from typing import List, Tuple

# To plot pretty figures
import seaborn as sns
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc('axes', labelsize=14)
mpl.rc('xtick', labelsize=12)
mpl.rc('ytick', labelsize=12)

sns.set_theme()

In [3]:
# Project Paths
NOTEBOOK_NAME = "3.0-Surprisal"
PROJECT_ROOT_DIR = "."

# --- Result dirs. 
OUTPUT_DIR = os.path.join(PROJECT_ROOT_DIR,"outputs",NOTEBOOK_NAME)
os.makedirs(OUTPUT_DIR,exist_ok=True)


LOGGING_DIR = os.path.join(OUTPUT_DIR,"logs")
os.makedirs(LOGGING_DIR,exist_ok=True)

In [4]:
# Setup logging for this notebook 
from LLMDialogue.utils.logging import configure_logger, create_file_handler, create_stream_handler
from LLMDialogue.utils.general import get_system_device
import logging

configure_logger(
    handlers=[
        create_stream_handler(logging_level=logging.WARNING), 
        create_file_handler(
            outdir=LOGGING_DIR, 
            logging_level=logging.DEBUG, 
            reset_outdir=False # Keep logs across runs. 
        )
    ]
)


In [5]:

GLOBAL_DEVICE = get_system_device()
# Shared env. vars. 
GLOBAL_SEED = 42 
RESET_OUTDIR = True
# Configuring env.
if  GLOBAL_SEED is not None:
    # to make this notebook's output stable across runs
    np.random.seed(GLOBAL_SEED) 
    torch.manual_seed(GLOBAL_SEED)
# NOTE: Should be mps for mac. 
GLOBAL_DEVICE

device(type='mps')

In [6]:
#Reset all but the logs dir in the output folder
from LLMDialogue.utils.general import remove_dir, get_subdirs_in_dir
if RESET_OUTDIR:
    subdirs = get_subdirs_in_dir(OUTPUT_DIR)
    for subdir in subdirs:
        if not subdir in ['logs']:
            remove_dir(subdir)


## Datasets

### Speaker Identity Dataset

In [7]:
from LLMDialogue.datasets.speakerIdentity import (
    SpeakerIdentityDataSplits, 
    objectify as SIObjectify
)
from LLMDialogue.datasets import load

In [8]:
dset = load(
    dset_name="speakerIdentity",
    config_name="no_labels" # Labels is the one I want with GPT-2. 
)
dset_splits = SIObjectify(dset)
dset_splits.data_df_map['test']

,conversationName,conversationID,turnNumber,utterance
0,30b,0,0,i haven't seen the keys anywhere<ts>
1,30b,0,1,have you<ts>
2,101bf,1,0,did you get enough to eat<ts> why's that<ts>
3,113c,2,0,sorry that i'm preventing you from finishing t...
4,113c,2,1,no rush<ts>
...,...,...,...,...
1121,33c,746,1,that sucks<ts>
1122,45e,747,0,i have the secret santa gift in my bag<ts>
1123,45e,747,1,so far<ts>
1124,18bd,748,0,i can't find my phone<ts> since when<ts>


## Surprisal

### Method 1: Surprisal of End of Turn Token

In [9]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch.nn.functional import softmax
from LLMDialogue.models import GPT2, TurnGPT
from LLMDialogue.pipelines import CPPipeline, CPPipelineConfig

In [10]:
# Loading the simple huggingface models 

model = AutoModelForCausalLM.from_pretrained("gpt2")
tokenizer = AutoTokenizer.from_pretrained("gpt2")

In [11]:
# Taking the text of two turns (without any additional labels), can we generate 
# the surprisal of the end of turn token. 

turns = [
    "i haven't seen the keys anywhere", "have you"
]
turn = " ".join(turns)
turn 

"i haven't seen the keys anywhere have you"

In [12]:
# Now, tokenizer the turn and predict using the model
toks = tokenizer(turn,return_tensors="pt")
toks

{'input_ids': tensor([[  72, 4398,  470, 1775,  262, 8251, 6609,  423,  345]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [13]:
with torch.no_grad():
    # Output shape: batch_size, seq_len, vocab_size
    output = model(**toks)
logits = output['logits']
logits.shape


torch.Size([1, 9, 50257])

In [14]:
# We need to get the distribution for the words after the last word
seq_len = toks['input_ids'].shape[1]
last_logits = logits[-1,seq_len-1]
# Convert into probability distribution
last_logits_probs = softmax(last_logits,dim=-1)
last_logits_probs.sum()

tensor(1.0000)

In [15]:
# Get the probability of the end of word. 
eos_prob = last_logits_probs[tokenizer.eos_token_id]
# Get the surprisal, which is the negative log of the conditional prob. of the
# word. 
eos_surprisal = -1 * torch.log(eos_prob)
eos_prob, eos_surprisal

(tensor(2.5526e-05), tensor(10.5758))

In [16]:
gpt2 = GPT2(GLOBAL_SEED)
gpt2.load()

2024-05-28 14:27:21,537 | WARNING | gpt2 | WARNING: Loading pretrained model gpt2 and tokenizer gpt2 with default arguments.
This is equivalent to using the specified model and tokenizer directly from the transformers library.


In [17]:
utts = [
            "<START>",
            "<SP1> i haven't seen the keys anywhere <SP1>",
            "<SP2> have you <SP2>",
            "<END>"
        ]
toks = gpt2.encode(utts)
gpt2.decode(toks['input_ids'])

"<START> <SP1> i haven't seen the keys anywhere <SP1> <SP2> have you <SP2> <END>"

In [18]:
pipe_configs = CPPipelineConfig(
    model=gpt2, 
    utterances=utts, 
    N=-1,
    context_buffer_size=512,
    probability_method="eos"
)

pipe = CPPipeline(pipe_configs)
pipe.run()


Processing turns: 100%|██████████| 4/4 [00:01<00:00,  3.74it/s]


In [19]:
pipe.info().data

[{'turn_no': 1,
  'word_no': 0,
  'context': '<START> ',
  'word': '<SP1>',
  'last_word_prob': 0.00043384882155805826},
 {'turn_no': 1,
  'word_no': 1,
  'context': '<START> <SP1>',
  'word': 'i',
  'last_word_prob': 3.8645601307507604e-05},
 {'turn_no': 1,
  'word_no': 2,
  'context': '<START> <SP1> i',
  'word': "haven't",
  'last_word_prob': 5.208856691751862e-06},
 {'turn_no': 1,
  'word_no': 3,
  'context': "<START> <SP1> i haven't",
  'word': 'seen',
  'last_word_prob': 1.3542133274313528e-05},
 {'turn_no': 1,
  'word_no': 4,
  'context': "<START> <SP1> i haven't seen",
  'word': 'the',
  'last_word_prob': 6.934596513019642e-06},
 {'turn_no': 1,
  'word_no': 5,
  'context': "<START> <SP1> i haven't seen the",
  'word': 'keys',
  'last_word_prob': 5.4460815590573475e-05},
 {'turn_no': 1,
  'word_no': 6,
  'context': "<START> <SP1> i haven't seen the keys",
  'word': 'anywhere',
  'last_word_prob': 0.0005915844230912626},
 {'turn_no': 1,
  'word_no': 7,
  'context': "<START> <SP1>